# 🧠 CSC 588 — Data Mining Project Demo

**Author**: Fares Alotaibi
**Course**: CSC 588 — Data Warehousing & Mining

This single notebook runs **the entire project** end-to-end:

| Section | Track | Algorithms | Dataset |
|---|---|---|---|
| **A** | Setup | — | — |
| **B** | Data Exploration | — | Online Shoppers |
| **C** | Classification | **10** algorithms | Online Shoppers |
| **D** | Clustering | **3** algorithms | CC General |
| **E** | Association Rule Mining | **2** algorithms | Online Retail II |
| **F** | Interactive UI | Gradio app | All three |
| **G** | Hypothesis Evaluation | — | — |

**Total expected runtime**: 15–25 minutes (mostly Section C — classification).

---


## 🅰️ Section A — Setup

Clone the repository fresh and install dependencies. The `!rm -rf` ensures a clean clone even on re-runs.


In [ ]:
# Force a fresh clone so the latest source code is always used
!rm -rf Data_Mining_HW2
!git clone https://github.com/faresalotaibi888-gif/Data_Mining_HW2.git
%cd Data_Mining_HW2
!pip install -q -r requirements.txt


In [ ]:
# Import every module we'll use across the demo
import sys
sys.path.append('src')

# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Our project modules
from data_loader import load_online_shoppers, load_cc_general, load_online_retail
from preprocessing import (
    prepare_online_shoppers,
    describe_prepared_data,
    preprocess_cc_general,
    preprocess_online_retail,
    build_shoppers_preprocessor,
)
from classification import run_classification_benchmark
from clustering import (
    kmeans_diagnostics,
    suggest_dbscan_eps,
    run_kmeans,
    run_agglomerative,
    run_dbscan,
    compute_cluster_metrics,
    metrics_to_dataframe,
    profile_clusters,
)
from association import (
    mine_frequent_itemsets,
    mine_rules,
    generate_all_association_outputs,
)
from evaluation import (
    plot_metric_comparison,
    plot_time_vs_accuracy,
    plot_roc_curves,
    plot_confusion_matrices_grid,
    plot_feature_importance,
    plot_elbow,
    plot_silhouette_scores,
    plot_dendrogram,
    plot_dbscan_kdistance,
    plot_clusters_2d,
    save_results_table,
)

print("All modules imported.")


---
## 🅱️ Section B — Data Exploration (Online Shoppers)

Quick look at the classification dataset: shape, columns, target class imbalance.


In [ ]:
shoppers = load_online_shoppers()
print(f"Shape: {shoppers.shape}")
shoppers.head()


In [ ]:
# Class balance check
import seaborn as sns
counts = shoppers['Revenue'].value_counts()
print(f"Non-buyers: {counts[False]:,} ({counts[False]/len(shoppers):.1%})")
print(f"Buyers    : {counts[True]:,} ({counts[True]/len(shoppers):.1%})")

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=[str(i) for i in counts.index], y=counts.values,
            hue=[str(i) for i in counts.index],
            palette=['#e74c3c', '#2ecc71'], legend=False, ax=ax)
ax.set_title('Online Shoppers — Class distribution of Revenue')
ax.set_xlabel('Revenue')
ax.set_ylabel('Sessions')
plt.tight_layout()
plt.show()


**Observation**: Only ~15% of sessions end in a purchase — strong class imbalance. This motivates the **SMOTE oversampling** applied during preprocessing and is the basis of **Hypothesis H3**.

---
## 🅲 Section C — Classification (10 algorithms)

### C.1 — Preprocess

The pipeline:
1. Drop `Revenue`, cast to int target
2. One-hot encode 6 categorical columns; scale 10 numeric columns
3. Stratified 80/20 train/test split
4. **SMOTE on training set only** (test set keeps real-world imbalance)


In [ ]:
prepared = prepare_online_shoppers(shoppers)
describe_prepared_data(prepared)


### C.2 — Run the 10-algorithm benchmark

Each algorithm is tuned via `GridSearchCV` with stratified 5-fold cross-validation, then evaluated on the held-out test set.

**Algorithms**: Logistic Regression · KNN · Decision Tree · Random Forest · SVM (RBF) · Gaussian NB · LDA · Gradient Boosting · AdaBoost · MLP

**Expected runtime**: 8-15 minutes. SVM is the slowest.


In [ ]:
result = run_classification_benchmark(
    prepared.X_train,
    prepared.X_test,
    prepared.y_train,
    prepared.y_test,
)
result.results_df.round(4)


### C.3 — Save results table for the report

Saved to `results/tables/classification_results.csv`.


In [ ]:
save_results_table(result.results_df, 'classification_results')


### C.4 — Comparison figures

Each figure is saved with its exact filename — use these names verbatim when inserting screenshots into the report.

| Figure | Filename |
|---|---|
| Metrics comparison | `classification_metrics_comparison.png` |
| Time vs accuracy   | `classification_time_vs_accuracy.png`   |
| ROC curves         | `classification_roc_curves.png`         |
| Confusion matrices | `classification_confusion_matrices.png` |


In [ ]:
# Bar chart comparing F1 across all algorithms
fig = plot_metric_comparison(
    result.results_df, metric='f1',
    filename='classification_metrics_comparison',
    title='Classification — F1 Score Comparison',
)
plt.show()


In [ ]:
# Trade-off: training time vs test accuracy
fig = plot_time_vs_accuracy(
    result.results_df,
    filename='classification_time_vs_accuracy',
)
plt.show()


In [ ]:
# Overlay of ROC curves
fig = plot_roc_curves(
    result.fitted_models,
    prepared.X_test,
    prepared.y_test,
    filename='classification_roc_curves',
)
plt.show()


In [ ]:
# Grid of confusion matrices
fig = plot_confusion_matrices_grid(
    result.fitted_models,
    prepared.X_test,
    prepared.y_test,
    filename='classification_confusion_matrices',
)
plt.show()


### C.5 — Best model analysis

Pull out the top model by F1 score for the discussion section of the report.


In [ ]:
best_row = result.results_df.sort_values('f1', ascending=False).iloc[0]
best_name = best_row['algorithm']
best_model = result.fitted_models[best_name]
print(f"🏆 Best model: {best_name}")
print(f"   F1     = {best_row['f1']:.4f}")
print(f"   ROC-AUC= {best_row['roc_auc']:.4f}")
print(f"   Best params: {result.best_params[best_name]}")


In [ ]:
# Feature importance for the winning model, if supported
if hasattr(best_model, 'feature_importances_') or hasattr(best_model, 'coef_'):
    fig = plot_feature_importance(
        best_model,
        feature_names=prepared.feature_names,
        filename='classification_feature_importance',
        top_n=20,
        title=f'{best_name} — Top 20 Most Important Features',
    )
    plt.show()
else:
    print(f"{best_name} does not expose feature importances directly.")


---
## 🅳 Section D — Clustering (3 algorithms)

### D.1 — Upload the CC General dataset

The CC General dataset is hosted on Kaggle (requires authentication). Download it once from https://www.kaggle.com/datasets/arjunbhasin2013/ccdata and run the cell below — Colab will prompt you to upload the CSV.


In [ ]:
from google.colab import files
import shutil
from pathlib import Path

print("Please select CC_GENERAL.csv from your computer...")
uploaded = files.upload()

Path("data").mkdir(exist_ok=True)
for filename in uploaded.keys():
    shutil.move(filename, f"data/{filename}")
    print(f"Saved → data/{filename}")


In [ ]:
cc = load_cc_general()
print(f"Shape: {cc.shape}")
cc.head()


### D.2 — Preprocess

Drop `CUST_ID`, median-impute missing values, standardise every feature.


In [ ]:
X_cc, cc_feature_names, cc_scaler = preprocess_cc_general(cc)
print(f"Scaled feature matrix: {X_cc.shape}")


### D.3 — K-Means diagnostics

Inertia (within-cluster sum of squares) and silhouette score for k=2..10.

Saved to `clustering_elbow_kmeans.png` and `clustering_silhouette_kmeans.png`.


In [ ]:
km_diag = kmeans_diagnostics(X_cc, k_min=2, k_max=10)

fig = plot_elbow(km_diag.k_values, km_diag.inertias,
                 filename='clustering_elbow_kmeans')
plt.show()

fig = plot_silhouette_scores(km_diag.k_values, km_diag.silhouettes,
                             filename='clustering_silhouette_kmeans')
plt.show()


### D.4 — DBSCAN k-distance for eps selection

Saved to `clustering_dbscan_kdistance.png`.


In [ ]:
kdist, suggested_eps = suggest_dbscan_eps(X_cc, k=5)
print(f"Suggested eps from knee point: {suggested_eps:.3f}")

fig = plot_dbscan_kdistance(kdist, suggested_eps,
                            filename='clustering_dbscan_kdistance', k=5)
plt.show()


### D.5 — Dendrogram (Agglomerative)

Saved to `clustering_dendrogram.png`.


In [ ]:
fig = plot_dendrogram(X_cc, filename='clustering_dendrogram',
                       method='ward', max_display=30)
plt.show()


### D.6 — Fit all three algorithms

Hyperparameter choices come from the diagnostics above.


In [ ]:
# K-Means with k chosen near the elbow / silhouette peak
labels_km, model_km = run_kmeans(X_cc, n_clusters=4)

# Agglomerative — same k for fair comparison
labels_ag, model_ag = run_agglomerative(X_cc, n_clusters=4)

# DBSCAN — eps from the k-distance plot
labels_db, model_db = run_dbscan(X_cc, eps=suggested_eps, min_samples=10)

# Compute metrics for each
metrics_list = [
    compute_cluster_metrics(X_cc, labels_km, algorithm_name='K-Means'),
    compute_cluster_metrics(X_cc, labels_ag, algorithm_name='Agglomerative'),
    compute_cluster_metrics(X_cc, labels_db, algorithm_name='DBSCAN'),
]
clustering_results = metrics_to_dataframe(metrics_list)
save_results_table(clustering_results, 'clustering_results')
clustering_results.round(3)


### D.7 — 2D PCA visualisation

Project clusters into 2D for visual comparison. Three figures saved:
`clustering_pca_kmeans.png`, `clustering_pca_agglomerative.png`, `clustering_pca_dbscan.png`.


In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_cc)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.2%}")

fig = plot_clusters_2d(X_2d, labels_km, filename='clustering_pca_kmeans',
                      title='K-Means (k=4)')
plt.show()

fig = plot_clusters_2d(X_2d, labels_ag, filename='clustering_pca_agglomerative',
                      title='Agglomerative (k=4, Ward linkage)')
plt.show()

fig = plot_clusters_2d(X_2d, labels_db, filename='clustering_pca_dbscan',
                      title=f'DBSCAN (eps={suggested_eps:.2f})')
plt.show()


### D.8 — Profile K-Means clusters

Mean of every original feature within each cluster — used to describe segments in plain English in the report.

Saved to `results/tables/clustering_profile_kmeans.csv`.


In [ ]:
# Use original (unscaled) data for interpretable units
cc_for_profile = cc.drop(columns=['CUST_ID'], errors='ignore').fillna(cc.median(numeric_only=True))
profile = profile_clusters(cc_for_profile, labels_km, feature_names=cc_feature_names)
save_results_table(profile, 'clustering_profile_kmeans')
profile.round(2)


---
## 🅴 Section E — Association Rule Mining

### E.1 — Load Online Retail II

Auto-downloaded from UCI (~50 MB Excel file). Loading takes ~30 seconds.


In [ ]:
retail = load_online_retail()
print(f"Shape: {retail.shape}")
retail.head()


### E.2 — Reshape to basket form (UK transactions only)

We filter to one country to keep mining tractable. Output is a 0/1 invoice × product matrix.


In [ ]:
basket = preprocess_online_retail(retail, country='United Kingdom', min_basket_size=2)
print(f"Basket matrix: {basket.shape[0]:,} invoices × {basket.shape[1]:,} products")


### E.3 — Mine frequent itemsets — Apriori vs FP-Growth

Both run on the same basket, so frequent itemsets are identical. The interesting metric is **runtime**: FP-Growth should be substantially faster.


In [ ]:
apriori_itemsets, fp_itemsets, runtimes = mine_frequent_itemsets(
    basket, min_support=0.03,
)
print(f"\nApriori   itemsets: {len(apriori_itemsets):,}")
print(f"FP-Growth itemsets: {len(fp_itemsets):,}")
print(f"Speedup (Apriori / FP-Growth): {runtimes['Apriori'] / runtimes['FP-Growth']:.1f}×")


### E.4 — Generate association rules


In [ ]:
rules = mine_rules(fp_itemsets, min_confidence=0.5, top_n=None)
print(f"Total rules: {len(rules)}")
rules.head(10)


### E.5 — All comparison figures + save tables

Saves figures:
* `association_runtime_comparison.png`
* `association_top_rules_by_lift.png`
* `association_support_confidence_scatter.png`


In [ ]:
paths = generate_all_association_outputs(
    apriori_itemsets, fp_itemsets, runtimes, rules,
)


---
## 🅵 Section F — Interactive Gradio UI

Launches a 4-tab web application with all three models accessible interactively.

* **🔮 Classification**: enter session features → predict purchase probability
* **🎯 Clustering**: explore each cluster's characteristics
* **🛒 Association**: search rules by product keyword

Colab will print a **public `gradio.live` URL** — share that link to show the UI to anyone during the class demo.


In [ ]:
from ui_app import launch_app

# Build a one-row template DataFrame so the UI can transform user input
# in the same shape the classifier expects.
classification_template = shoppers.drop(columns=['Revenue']).head(1)

launch_app(
    classification_models=result.fitted_models,
    classification_preprocessor=prepared.preprocessor,
    classification_feature_template=classification_template,
    clustering_models={
        'K-Means': labels_km,
        'Agglomerative': labels_ag,
        'DBSCAN': labels_db,
    },
    clustering_X_scaled=X_cc,
    clustering_feature_names=cc_feature_names,
    clustering_scaler=cc_scaler,
    association_rules_df=rules,
    share=True,
)


---
## 🅶 Section G — Hypothesis Evaluation Summary

From the Introduction, we set six testable hypotheses. The cells in this notebook produced the evidence — here's how each maps to the results.

| # | Hypothesis | Evidence | Verdict |
|---|---|---|---|
| **H1** | Ensembles outperform single-model methods | Section C.2 results table — RF / GB / AdaBoost vs LR / DT / NB | (fill in after run) |
| **H2** | Tree ensembles offer best speed/accuracy trade-off | Section C.4 — `classification_time_vs_accuracy.png` | (fill in) |
| **H3** | SMOTE improves minority recall | Compare recall column with/without SMOTE | (fill in) |
| **H4** | DBSCAN captures non-spherical structure | Section D.7 — PCA plots; DBSCAN clusters take irregular shapes vs K-Means' spheres | (fill in) |
| **H5** | FP-Growth substantially faster than Apriori; same rules | Section E.3 — speedup × in printout | (fill in) |
| **H6** | Tuning improves over defaults for majority of algorithms | Section C — CV F1 vs default-F1 (we'd need a baseline run) | (fill in) |

---

### 📂 Complete file inventory for the report

```
results/tables/
├── classification_results.csv
├── clustering_results.csv
├── clustering_profile_kmeans.csv
├── association_frequent_itemsets.csv
└── association_rules.csv

results/figures/
├── classification_metrics_comparison.png
├── classification_time_vs_accuracy.png
├── classification_roc_curves.png
├── classification_confusion_matrices.png
├── classification_feature_importance.png
├── clustering_elbow_kmeans.png
├── clustering_silhouette_kmeans.png
├── clustering_dbscan_kdistance.png
├── clustering_dendrogram.png
├── clustering_pca_kmeans.png
├── clustering_pca_agglomerative.png
├── clustering_pca_dbscan.png
├── association_runtime_comparison.png
├── association_top_rules_by_lift.png
└── association_support_confidence_scatter.png
```

All figure filenames match exactly the names used in the .py source and in this notebook. **Use these names verbatim in your report**.

---

### 🎉 Demo complete

Open the Gradio link from Section F to wrap up the live demo with the interactive UI.
